# M3C4 — Level 2: Guided Feature Selection Practice

**Purpose:** implement correlation filtering, MI selection, RFE and a fair model comparison.

**Do:** save a copy in Drive. Run from top to bottom; upload both `telco_source.csv` and `M3C4_Runtime.zip` when asked. Replace each `raise NotImplementedError` with your code. The short error means the exercise is still unfinished; complete the function and rerun the check.

**Check:** choose `top_k` and write your reason before validation. Write your own observations and conclusion. The final test stays closed. After a runtime reset start again at C01.

## C01 — Start the workspace

**Purpose:** rebuild the protected split and train-fitted preprocessing.

**Do:** run the cell; select both issued files if asked.

**Check:** train 4225, validation 1409, final test reserved 1409, all 50 columns. IDs and target are excluded from predictors.

In [ ]:
#@title ▶ Start the lesson
from pathlib import Path
from zipfile import ZipFile, BadZipFile
comparison = results = rfe_columns = None  # Invalidate old results before a new start.
required = ['telco_source.csv', 'M3C4_Runtime.zip']
if any(not Path(name).exists() for name in required):
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Place telco_source.csv and M3C4_Runtime.zip beside this notebook, then rerun C01.') from None
    print('Select BOTH telco_source.csv and M3C4_Runtime.zip from the lesson package.')
    files.upload()
missing = [name for name in required if not Path(name).exists()]
if missing:
    raise FileNotFoundError('Missing: ' + ', '.join(missing) + '. Rerun C01 with the issued files and original names.')
try:
    with ZipFile('M3C4_Runtime.zip') as support:
        folder = Path('_m3c4_runtime'); folder.mkdir(exist_ok=True)
        for name in ['m3c4_runtime.py', 'M3_Split_Card.json']:
            (folder / name).write_bytes(support.read(name))
except (BadZipFile, KeyError):
    raise ValueError('Wrong or damaged M3C4_Runtime.zip. Upload the issued ZIP again, then rerun C01.') from None
exec(compile((folder / 'm3c4_runtime.py').read_text(encoding='utf-8'), 'm3c4_runtime.py', 'exec'), globals())

## C02 — Find correlated pairs

**Purpose:** find strong numeric relationships on training data.

**Do:** Complete the function. Use the declared numeric order and return each pair once when abs(r) > 0.85. Hint: use `frame[columns].corr()` and nested loops over positions.

**Check:** The check compares the full pairs and coefficients against an independent calculation.

In [ ]:
def find_correlated_pairs(frame, columns, threshold=0.85):
    """Return DataFrame columns A, B, r in declared pair order."""
    # TODO: calculate correlations; collect (earlier, later, coefficient).
    raise NotImplementedError("C02 TODO: implement find_correlated_pairs, then rerun this cell and its check.")

In [ ]:
current_guard()
pairs = find_correlated_pairs(train_features, numeric)
expected_corr = train_features[numeric].corr()
expected_pairs = pd.DataFrame([(a,b,float(expected_corr.loc[a,b])) for i,a in enumerate(numeric) for b in numeric[i+1:] if abs(expected_corr.loc[a,b]) > .85], columns=['A','B','r'])
pd.testing.assert_frame_equal(pairs.reset_index(drop=True), expected_pairs, check_dtype=False)
show_table('Train correlation pairs', 'Each pair appears once; strong correlation does not establish causation.', pairs, {'A':'Feature A','B':'Feature B','r':'Correlation r'})

## C03 — Apply the declared rule

**Purpose:** retain the earlier source/formula field and avoid duplicate removals.

**Do:** For each pair in order, remove B only if neither member was removed already. Explain the reason for every pair in your own words. Return decisions and candidates; all 50 remain available for the baseline.

**Check:** The check tests decisions and the 47 candidates.

In [ ]:
def filter_pairs(pairs, all_columns):
    """Return (decisions DataFrame A,B,Removed,Reason; candidate column list)."""
    # TODO: keep a set of removed numeric names and a list of records.
    # Prefix removed numeric names with 'num__' when filtering all_columns.
    raise NotImplementedError("C03 TODO: implement filter_pairs, then rerun this cell and its check.")

In [ ]:
decisions, candidates = filter_pairs(pairs, list(X_train.columns))
removed_expected=set(); records=[]
for a,b,_ in expected_pairs.itertuples(index=False, name=None):
    if a in removed_expected or b in removed_expected:
        records.append((a,b,'already covered','A member was already removed; no extra removal'))
    else:
        removed_expected.add(b); records.append((a,b,b,'Keep earlier source/formula in declared order for interpretability'))
expected_decisions=pd.DataFrame(records,columns=['A','B','Removed','Reason'])
pd.testing.assert_frame_equal(decisions[['A','B','Removed']].reset_index(drop=True),expected_decisions[['A','B','Removed']],check_dtype=False)
assert decisions['Reason'].map(lambda value: isinstance(value,str) and len(value.strip()) >= 5).all(), 'C03: explain every decision in the Reason column using your own words.'
assert candidates == [c for c in X_train if c not in {'num__'+x for x in removed_expected}] and len(candidates)==47
show_table('Correlation decisions', 'The declared order determines which field stays. Previously covered pairs need no extra removal.', decisions, {'A':'Feature A','B':'Feature B','Removed':'Removed feature'})
show_note('Candidate count', f'All baseline: {len(X_train.columns)}; candidates for MI and RFE: {len(candidates)}.')

**My observation:** Name one pair, its correlation, removed field and why the other field stays: …

**Limit:** What can correlation alone not prove? …

## C04 — Rank candidates with MI

**Purpose:** measure each candidate’s information about the train target.

**Do:** Build the discrete mask: one-hot columns plus `num__SeniorCitizen` and `num__num_services`. Factorize those columns before calling `mutual_info_classif` with seed 42. Return a descending, stable Series indexed by feature.

**Check:** The check verifies the ranking and values against a separate calculation.

In [ ]:
def rank_mi(X, y, candidates):
    """Return descending MI Series, index = candidate names."""
    # TODO: build mask, copy candidate data, factorize discrete columns, calculate MI.
    raise NotImplementedError("C04 TODO: implement rank_mi, then rerun this cell and its check.")

In [ ]:
current_guard()
mi_scores=rank_mi(X_train,train_y,candidates)
mask=np.array([c.startswith('cat__') or c in ['num__SeniorCitizen','num__num_services'] for c in candidates])
mi_input=X_train[candidates].copy()
for c in np.array(candidates)[mask]: mi_input[c]=pd.factorize(mi_input[c],sort=True)[0]
expected_mi=pd.Series(mutual_info_classif(mi_input,train_y,discrete_features=mask,random_state=42),index=candidates).sort_values(ascending=False,kind='stable')
pd.testing.assert_series_equal(mi_scores,expected_mi,check_names=False)
show_note('Discrete columns', ', '.join(np.array(candidates)[mask]))
show_table('Train MI ranking', 'Higher MI indicates information about Churn on train; it does not guarantee the best combination.', mi_scores.rename_axis('Feature').reset_index(name='Train MI'),max_height=420)

## C05 — Choose before validation

**Purpose:** make a selection without using validation scores.

**Do:** choose an integer `top_k` from 1 to 47 and write a reason. This is the number of highest-ranked features, not a score threshold. Choose once before viewing validation.

**Check:** name one selected feature you can explain. If you change the choice, rerun C05 through C08; changing it after viewing scores makes the validation comparison exploratory.

In [ ]:
top_k = None
selection_reason = ''
# My selected feature and explanation: ...

## C06 — Select MI columns

**Purpose:** turn the train ranking into a feature list.

**Do:** Return the first `top_k` index names in order. Hint: use `scores.index[:k]`.

**Check:** The check rejects an empty choice and checks the exact ordered selection before any validation.

In [ ]:
def choose_mi(scores, k):
    """Return the first k feature names as a list."""
    # TODO: select the top k feature names.
    raise NotImplementedError("C06 TODO: implement choose_mi, then rerun this cell and its check.")

In [ ]:
assert isinstance(top_k,int) and not isinstance(top_k,bool) and 1<=top_k<=len(candidates), 'C05: set top_k to an integer from 1 to 47 before validation.'
assert isinstance(selection_reason,str) and selection_reason.strip(), 'C05: write your reason before validation.'
mi_columns=choose_mi(mi_scores,top_k)
assert mi_columns==list(mi_scores.index[:top_k]), 'C06: return the top k feature names in rank order.'
selection_stamp=(top_k,selection_reason,tuple(mi_columns))
show_table('Chosen MI features', 'Chosen before validation using the train ranking.', pd.DataFrame({'Feature':mi_columns}),max_height=360)

## C07 — Select with RFE

**Purpose:** repeatedly fit a model and eliminate one feature.

**Do:** Fit RFE on train candidates only, using RF100, seed 42, step 1, and floor(half the candidates), at least one. Return selected names and a table with Feature, Selected, Rank.

**Check:** The check fits an independent selector and verifies support and ranks. This step can take several minutes.

In [ ]:
def select_rfe(X, y, candidates):
    """Return (selected names, DataFrame Feature, Selected, Rank)."""
    # TODO: create RFE, fit train candidates, use support_ and ranking_.
    raise NotImplementedError("C07 TODO: implement select_rfe, then rerun this cell and its check.")

In [ ]:
current_guard()
assert selection_stamp==(top_k,selection_reason,tuple(mi_columns)), 'C05 choice changed: rerun C05–C07.'
start=time.perf_counter()
rfe_columns,rfe_table=select_rfe(X_train,train_y,candidates)
elapsed=time.perf_counter()-start
reference=RFE(RandomForestClassifier(n_estimators=100,random_state=42),n_features_to_select=max(1,len(candidates)//2),step=1).fit(X_train[candidates],train_y)
assert rfe_columns==list(np.array(candidates)[reference.support_])
pd.testing.assert_frame_equal(rfe_table.reset_index(drop=True),pd.DataFrame({'Feature':candidates,'Selected':reference.support_,'Rank':reference.ranking_}),check_dtype=False)
show_table('RFE selection and ranks', 'Rank 1 is selected; higher ranks were removed earlier.', rfe_table,max_height=420)
show_note('RFE summary',f'Kept {len(rfe_columns)} of {len(candidates)} candidates. Student selector runtime: {elapsed:.2f} seconds.\nMI ∩ RFE: '+', '.join(sorted(set(mi_columns)&set(rfe_columns))))

## C08 — Compare all, MI and RFE

**Purpose:** compare feature sets on the same validation customers.

**Do:** For each set fit RF100 seed 42 on train, predict validation, and return Feature Set, Num Features, Accuracy and F1 (positive label Yes). Hint: loop over a list of label/columns pairs.

**Check:** The check independently calculates all three rows. Keep the choice made before validation.

In [ ]:
def compare_sets(X_train, X_validation, y_train, y_validation, mi_columns, rfe_columns):
    """Return a DataFrame for all, MI and RFE with counts, Accuracy and F1."""
    # TODO: fit and score one model per set.
    raise NotImplementedError("C08 TODO: implement compare_sets, then rerun this cell and its check.")

In [ ]:
current_guard()
assert selection_stamp==(top_k,selection_reason,tuple(mi_columns)), 'Choice changed: rerun C05–C08.'
results=compare_sets(X_train.copy(),X_validation.copy(),train_y.copy(),validation_y.copy(),mi_columns,rfe_columns)
expected=pd.DataFrame([rf_row('all',list(X_train)),rf_row('MI',mi_columns),rf_row('RFE',rfe_columns)])
pd.testing.assert_frame_equal(results.reset_index(drop=True),expected,check_dtype=False,atol=1e-12,rtol=1e-12)
show_table('Validation comparison', 'Same 1409 customers and RF100 settings; compare counts, Accuracy and F1 together.', results)
show_note('Interpret the gap',f'Largest accuracy gap corresponds to approximately {round((results.Accuracy.max()-results.Accuracy.min())*1409)} customer predictions.')

## C09 — Explain and submit

**Purpose:** connect the measured results to a recommendation.

**Do:** answer using your actual output, then save the executed notebook in Drive or download it.

**Check:** reopen it and confirm your completed code, tables and own answers are present. The final test remains closed.

1. Which set would you recommend? Cite its feature count, Accuracy and F1 and explain your trade-off: …
2. Which features occur in both MI and RFE? Name one disagreement if there is one: …
3. How many customers correspond to the largest accuracy gap? Why does F1 also matter? …
4. What can the selection say about MonthlyCharges, and why does it not prove that charges cause churn? …
5. For a new dataset, what must be fitted again on train, and when would you choose `top_k`? …